In [5]:
import os

# --- CAMBIÁ ESTO POR LA RUTA DE TU ARCHIVO ---
RUTA_ENTRADA = r"C:\Users\gabri\Downloads\30-71113382-4_2025-12_0_SAC_LSD (4).txt"
RUTA_SALIDA = RUTA_ENTRADA.replace(".txt", "_AFIP_FINAL_V4.txt")

def reconstruir_archivo_afip():
    print(f"Reconstruyendo: {RUTA_ENTRADA}...")
    
    try:
        with open(RUTA_ENTRADA, 'r', encoding='latin-1') as f:
            lineas = f.readlines()
    except FileNotFoundError:
        print("Error: No se encuentra el archivo.")
        return

    nuevas_lineas = []
    
    # Contadores
    count_03 = 0
    count_04 = 0
    vistos = set()

    # --- 1. PROCESAR CABECERA ORIGINAL ---
    if len(lineas) > 0 and lineas[0].startswith('01'):
        cabecera_raw = lineas[0].strip()
        # Tomamos los datos fijos (CUIT, Periodo, etc). 
        # Asumimos que los primeros 32 caracteres son los datos fijos.
        datos_cabecera = cabecera_raw[:32] 
        lineas_cuerpo = lineas[1:]
    else:
        # Si no hay cabecera, inventamos una dummy (peligroso, pero necesario si está corrupto)
        datos_cabecera = "" 
        lineas_cuerpo = lineas

    # --- 2. PROCESAR CUERPO ---
    for linea in lineas_cuerpo:
        linea = linea.strip('\n').strip('\r') # Limpieza total
        if not linea: continue
        
        tipo = linea[0:2]

        if tipo == '03':
            count_03 += 1
            
            # --- CORRECCIÓN REGISTRO 03 (EMPLEADOS) ---
            # Regla: Debe medir 370 chars. Relleno con '0'.
            # Primero, aseguramos la unificación de bases (tu pedido original)
            if len(linea) >= 153:
                rem_9 = linea[138:153]
                # Reconstruimos bases 1 a 5 y 8 con la Rem 9
                linea = (
                    linea[0:33] + 
                    rem_9 * 5 +    # Rem 1,2,3,4,5
                    linea[108:123] + # Rem 6,7
                    rem_9 +        # Rem 8
                    linea[138:]
                )
            
            # AHORA EL RELLENO FINAL CON CEROS
            if len(linea) < 370:
                faltan = 370 - len(linea)
                linea = linea + ('0' * faltan)
            
            nuevas_lineas.append(linea)

        elif tipo == '04':
            # --- CORRECCIÓN REGISTRO 04 (CONCEPTOS) ---
            # Regla: Debe medir 51 chars EXACTOS. Relleno con ' '.
            
            # Filtro duplicados
            clave = linea[2:23] 
            if clave in vistos: continue
            vistos.add(clave)
            
            count_04 += 1
            
            # Limpiamos espacios a la derecha por si acaso viene sucio
            linea_limpia = linea.rstrip()
            
            # RELLENO FINAL CON ESPACIOS
            # ljust(51) agrega espacios hasta llegar a 51.
            linea_final = linea_limpia.ljust(51, ' ')
            
            # Seguridad extra: Si mide más de 51, cortamos (raro, pero posible)
            if len(linea_final) > 51:
                linea_final = linea_final[:51]
                
            nuevas_lineas.append(linea_final)
            
        else:
            nuevas_lineas.append(linea)

    # --- 3. RECONSTRUIR CABECERA (REGISTRO 01) ---
    # Formato Header: Datos(32) + Cant03(6) + Cant04(6) + Relleno(Espacios) = 83 Total
    
    str_03 = str(count_03).zfill(6)
    str_04 = str(count_04).zfill(6)
    
    if len(datos_cabecera) >= 32:
        # Usamos los datos originales
        header_nuevo = datos_cabecera[:32] + str_03 + str_04
    else:
        # Si la cabecera original estaba rota (<32), usamos lo que haya y rezamos, 
        # o rellenamos con espacios si faltaba CUIT/Periodo (muy raro).
        header_nuevo = datos_cabecera + str_03 + str_04
        
    # RELLENO FINAL CABECERA CON ESPACIOS (Hasta 83 chars estándar)
    header_final = header_nuevo.ljust(83, ' ')
    
    nuevas_lineas.insert(0, header_final)

    # --- 4. GUARDAR BINARIO ---
    # Escribimos en modo binario o forzando \r\n para que Windows no haga lío
    with open(RUTA_SALIDA, 'w', encoding='latin-1', newline='\r\n') as f_out:
        for l in nuevas_lineas:
            f_out.write(l + '\n')

    print("="*30)
    print("ARCHIVO GENERADO:", os.path.basename(RUTA_SALIDA))
    print(f"Empleados (03): {count_03} (Rellenados con 0 hasta 370)")
    print(f"Conceptos (04): {count_04} (Rellenados con espacios hasta 51)")
    print("="*30)

if __name__ == "__main__":
    reconstruir_archivo_afip()

Reconstruyendo: C:\Users\gabri\Downloads\30-71113382-4_2025-12_0_SAC_LSD (4).txt...
ARCHIVO GENERADO: 30-71113382-4_2025-12_0_SAC_LSD (4)_AFIP_FINAL_V4.txt
Empleados (03): 865 (Rellenados con 0 hasta 370)
Conceptos (04): 174 (Rellenados con espacios hasta 51)
